In [4]:
from pyspark.sql import functions as F, Window
from spark_session import get_spark
import sources

In [5]:
spark = get_spark("windows")
orders = sources.orders(spark).where("quantity > 0 AND customer_id IS NOT NULL").dropDuplicates()

by_customer = Window.partitionBy("customer_id").orderBy("order_ts")
numbered = (
    orders
    .withColumn("nth", F.row_number().over(by_customer))
    .withColumn("prev_ts", F.lag("order_ts").over(by_customer))
    .withColumn("gap_h", F.round((F.col("order_ts").cast("long") - F.col("prev_ts").cast("long")) / 3600, 1))
)
numbered.where("customer_id = 7").select("customer_id", "order_ts", "nth", "gap_h").show(4)
firsts = numbered.where("nth = 1")
print("customers with a first order: ", firsts.count())


+-----------+-------------------+---+-----+
|customer_id|           order_ts|nth|gap_h|
+-----------+-------------------+---+-----+
|          7|2026-10-01 00:07:46|  1| NULL|
|          7|2026-10-01 04:25:17|  2|  4.3|
|          7|2026-10-01 05:46:57|  3|  1.4|
|          7|2026-10-01 08:01:20|  4|  2.2|
+-----------+-------------------+---+-----+
only showing top 4 rows
customers with a first order:  2000


In [6]:
sales = (
    orders.join(F.broadcast(sources.products(spark)), "product_id")
    .withColumn("revenue", F.col("quantity") * F.col("price"))
)
daily = sales.groupBy(F.to_date("order_ts").alias("day")).agg(F.round(F.sum("revenue"), 2).alias("revenue"))
running = Window.orderBy("day").rowsBetween(Window.unboundedPreceding, Window.currentRow)
daily.withColumn("to_date", F.round(F.sum("revenue").over(running), 2)).orderBy("day").show(4)

by_cat = Window.partitionBy("category").orderBy(F.desc("revenue"))
per_product = sales.groupBy("category", "product_id").agg(F.round(F.sum("revenue"), 2).alias("revenue"))
(per_product.withColumn("rank", F.dense_rank().over(by_cat))
             .where("rank <= 2").orderBy("category", "rank").show())


+----------+----------+-------------+
|       day|   revenue|      to_date|
+----------+----------+-------------+
|2026-10-01|8563117.67|   8563117.67|
|2026-10-02|8447642.49|1.701076016E7|
|2026-10-03|8604813.33|2.561557349E7|
|2026-10-04|8760294.44|3.437586793E7|
+----------+----------+-------------+
only showing top 4 rows
+-----------+----------+------------+----+
|   category|product_id|     revenue|rank|
+-----------+----------+------------+----+
|      Books|      P013|   780492.24|   1|
|      Books|      P038|   737662.08|   2|
|Electronics|      P018|1.19518744E7|   1|
|Electronics|      P024|1.04159051E7|   2|
|       Home|      P020|  3352655.06|   1|
|       Home|      P009|  3213379.51|   2|
|     Sports|      P026|  3386678.88|   1|
|     Sports|      P040|  2301123.06|   2|
+-----------+----------+------------+----+



In [1]:
spark.stop()

NameError: name 'spark' is not defined